In [0]:
from pyspark.sql.functions import col, coalesce, expr

df = spark.table("new.default.suzuki_stock")

clean_df = df \
    .withColumn("Date",
        coalesce(
            expr("try_to_date(Date, 'M/d/yyyy')"),
            expr("try_to_date(Date, 'MM-dd-yyyy')"),
            expr("try_to_date(Date, 'yyyy-MM-dd')")
        )
    ) \
    .withColumn("Open", col("Open").cast("double")) \
    .withColumn("High", col("High").cast("double")) \
    .withColumn("Low", col("Low").cast("double")) \
    .withColumn("Close", col("Close").cast("double")) \
    .withColumn("Volume", col("Volume").cast("long")) \
    .dropDuplicates(["Date"])

clean_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("new.default.suzuki_stock_silver")

display(clean_df)